# Tool 7 — Manually reject flagged epochs (Phase 2b)

Inspect the epochs that **6_preprocessing** flagged, decide **keep / reject** for each, and export a
validated `*_clean-epo.fif`. Reads one participant's `*_all-epo.fif` (+ optional
`*_preprocessing_params.json`) from a *derivatives* root — the raw EDF is never reloaded (EEG-only,
whatever channels the `.fif` holds).

**Workflow:** ① Load a participant → ② Global per-stage report → ③ Navigate rejected epochs →
④ Override decisions & save. Tool-6 outputs are never modified.

In [ ]:
try:
    import os, sys, json, io
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    from ipyfilechooser import FileChooser

    # Make the shared library importable whether Voila is launched from the repo root or tools/.
    _here = os.getcwd()
    for _cand in (_here, os.path.join(_here, 'tools')):
        if os.path.isfile(os.path.join(_cand, 'qc_rejected_epochs_lib.py')):
            if _cand not in sys.path:
                sys.path.insert(0, _cand)
            break
    import qc_rejected_epochs_lib as L
except ImportError as e:
    print("⚠️ Import error — a package is missing:", e)
else:
    print("✅ Packages imported successfully!")

# Shared mutable state across sections.
S = {}

# Output areas (one per section).
out_load   = widgets.Output()
out_report = widgets.Output()
out_nav    = widgets.Output()
out_review = widgets.Output()

def show_fig(fig):
    # Render a matplotlib Figure as a PNG widget — reliable under Voila with the Agg backend
    # (plain display(fig) would only print the figure's text repr). Same pattern as tools 5/8.
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    plt.close(fig)
    display(widgets.Image(value=buf.getvalue(), format='png'))

if not L.HAS_SPECPARAM:
    print('WARNING: specparam not available — 1/f metrics will be blank.')


## Section 1 — Load a participant

In [ ]:
fc_deriv = FileChooser(os.getcwd())
fc_deriv.show_only_dirs = True
fc_deriv.title = '<b>Derivatives root</b> (folder holding the tool-6 <code>*_all-epo.fif</code> files):'

dd_part    = widgets.Dropdown(description='Participant:', options=[], style={'description_width':'110px'},
                              layout=widgets.Layout(width='340px'))
txt_custom = widgets.Text(description='Custom stages:', value='', style={'description_width':'110px'},
                          layout=widgets.Layout(width='340px'),
                          placeholder='e.g. N4  (comma-separated, optional)')
lbl_scan   = widgets.HTML('<i>Select a derivatives root.</i>')
btn_load   = widgets.Button(description='Load participant', button_style='primary', icon='folder-open')

def _on_folder(chooser):
    try:
        folder = fc_deriv.selected_path
        if not folder:
            return
        parts = L.find_participants(folder)
        S['parts'] = {p['file_id']: p for p in parts}
        S['deriv_root'] = folder
        dd_part.options = [p['file_id'] for p in parts]
        cs = L.load_custom_stages(folder)
        txt_custom.value = ','.join(cs)
        colour = '#2e7d32' if parts else '#c62828'
        lbl_scan.value = f'<span style="color:{colour}">{len(parts)} participant(s) with tool-6 outputs found.</span>'
    except Exception as e:
        lbl_scan.value = f'<span style="color:#c62828">Scan error: {e}</span>'

fc_deriv.register_callback(_on_folder)

def _on_load(b):
    with out_load:
        clear_output(wait=True)
        try:
            fid = dd_part.value
            if not fid:
                print('Select a participant first.'); return
            p = S['parts'][fid]
            P = L.load_participant(p['fif'])
            cs = L.parse_custom_field(txt_custom.value)
            thr, info = L.load_params(p['folder'], fid, custom_stages_fallback=cs)
            S.update({'P': P, 'fid': fid, 'folder': p['folder'], 'thr0': thr, 'pinfo': info,
                      'custom_stages': cs, 'freqs': None, 'psds': None, 'metrics': None,
                      'final_reject': P['reject_flag'].copy(), 'overridden': set()})
            S['ctx'] = L.load_context_epochs(p['folder'], fid)
            _build_amp_widgets(thr, cs)
            _set_scalar_thresholds(thr)
            n = len(P['stages']); nrej = int(P['reject_flag'].sum())
            src = 'from params JSON' if info['found'] else 'DEFAULTS (no *_preprocessing_params.json found)'
            print(f'Loaded {fid}:  {n} epochs, {nrej} rejected ({100*nrej/n:.1f}%)')
            print(f'  channels = {P["ch_names"]}   |   sfreq = {P["sfreq"]:.0f} Hz   |   epoch length = {info["epoch_length_s"]} s')
            print(f'  methods present = {P["methods_present"]}')
            if S['ctx'] is not None:
                print(f'  context channels = {S["ctx"]["labels"]}  (toggle in Section 3)')
            else:
                print('  context channels = none (no *_context-epo.fif companion from tool 6)')
            print(f'  thresholds = {src}')
            if cs:
                print(f'  custom stages = {cs}')
            print('\nProceed to Section 2 (report) or Section 3 (navigator).')
        except Exception as e:
            print(f'Load error: {e}')

btn_load.on_click(_on_load)

display(widgets.VBox([fc_deriv, widgets.HBox([dd_part, txt_custom]), lbl_scan, btn_load, out_load]))


## Section 2 — Global per-stage report

Per sleep stage: **PSD overlay** (rejected epochs coloured by method over clean epochs + IQR band),
**metric distributions** (clean vs rejected with threshold lines), a **p-p vs gradient scatter**, and a
**stage × method rejection table**. Thresholds below are pre-filled from the participant's params (or
tool-6 defaults) and drive the reference lines only. *1/f fitting runs on every epoch — allow ~1 min for
a full night.*

In [ ]:
# Threshold widgets (reference lines; editable). Amplitude widgets are rebuilt per participant.
box_amp   = widgets.HBox([])
_amp_w    = {}
ft_flat   = widgets.FloatText(description='Flat <', value=1.0,   style={'description_width':'70px'}, layout=widgets.Layout(width='150px'))
ft_grad   = widgets.FloatText(description='Grad >', value=100.0, style={'description_width':'70px'}, layout=widgets.Layout(width='150px'))
ft_mae    = widgets.FloatText(description='MAE >',  value=0.15,  style={'description_width':'70px'}, layout=widgets.Layout(width='150px'))
ft_r2     = widgets.FloatText(description='R² <',   value=0.95,  style={'description_width':'70px'}, layout=widgets.Layout(width='150px'))

def _build_amp_widgets(thr, custom_stages):
    global _amp_w
    _amp_w = {}
    order = ['W', 'N1', 'N2', 'N3', 'R'] + [c for c in custom_stages if c not in ('W','N1','N2','N3','R')]
    items = []
    for st in order:
        v = float(thr['amplitude_ptp_uV'].get(st, 250.0))
        w = widgets.FloatText(description=st, value=v, style={'description_width':'40px'},
                              layout=widgets.Layout(width='120px'))
        _amp_w[st] = w
        items.append(w)
    box_amp.children = [widgets.HTML('<b>Amplitude p-p (µV) per stage:</b>')] + items

def _set_scalar_thresholds(thr):
    ft_flat.value = float(thr['flat_ptp_uV'])
    ft_grad.value = float(thr['gradient_uV_per_sample'])
    ft_mae.value  = float(thr['1f_mae_max'])
    ft_r2.value   = float(thr['1f_r2_min'])

def _collect_thresholds():
    return {
        'amplitude_ptp_uV': {st: float(w.value) for st, w in _amp_w.items()},
        'flat_ptp_uV': float(ft_flat.value),
        'gradient_uV_per_sample': float(ft_grad.value),
        '1f_mae_max': float(ft_mae.value),
        '1f_r2_min': float(ft_r2.value),
    }

prog_report = widgets.IntProgress(description='1/f fit:', min=0, max=1, value=0,
                                  layout=widgets.Layout(width='320px'))
btn_run_report  = widgets.Button(description='Build & show inline', button_style='primary', icon='chart-area')
btn_save_report = widgets.Button(description='Build & save HTML (no inline)', icon='save',
                                 layout=widgets.Layout(width='230px'))

def _on_run_report(b):
    with out_report:
        clear_output(wait=True)
        try:
            if 'P' not in S:
                print('Load a participant first (Section 1).'); return
            P = S['P']
            print('Computing Welch PSDs…')
            fit_fmin, fit_fmax = S['pinfo']['fit_range']
            if S['freqs'] is None:
                S['freqs'], S['psds'] = L.compute_psds(P['epochs'], fmax=fit_fmax)
            thr = _collect_thresholds()
            prog_report.max = len(P['stages']); prog_report.value = 0
            print('Fitting 1/f per epoch (this can take ~1 min for a full night)…')
            S['metrics'] = L.compute_epoch_metrics(P['data_uV'], S['freqs'], S['psds'], fmin=fit_fmin,
                                                   progress=lambda d, t: setattr(prog_report, 'value', d))
            figs, html = L.build_participant_report_figs(P, S['metrics'], S['freqs'], S['psds'],
                                                         thr, S['custom_stages'])
            clear_output(wait=True)
            display(HTML(html))
            for name, fig in figs:
                show_fig(fig)
            btn_save_report.disabled = False
        except Exception as e:
            print(f'Report error: {e}')

def _on_save_report(b):
    # Standalone: computes PSDs / 1-f metrics if Build was not clicked, then writes the HTML
    # without rendering anything inline (keeps the notebook light).
    with out_report:
        clear_output(wait=True)
        try:
            if 'P' not in S:
                print('Load a participant first (Section 1).'); return
            P, thr = S['P'], _collect_thresholds()
            fit_fmin, fit_fmax = S['pinfo']['fit_range']
            if S.get('freqs') is None:
                print('Computing Welch PSDs…')
                S['freqs'], S['psds'] = L.compute_psds(P['epochs'], fmax=fit_fmax)
            if S.get('metrics') is None:
                print('Fitting 1/f per epoch (this can take ~1 min for a full night)…')
                prog_report.max = len(P['stages']); prog_report.value = 0
                S['metrics'] = L.compute_epoch_metrics(P['data_uV'], S['freqs'], S['psds'], fmin=fit_fmin,
                                                       progress=lambda d, t: setattr(prog_report, 'value', d))
            figs, html = L.build_participant_report_figs(P, S['metrics'], S['freqs'], S['psds'],
                                                         thr, S['custom_stages'])
            out = Path(S['folder']) / f'{S["fid"]}_qc2b_report.html'
            L.save_report_html(out, f'{S["fid"]} — QC of rejected epochs', figs, html)
            print(f'Saved report → {out}')
        except Exception as e:
            print(f'Save error: {e}')

btn_run_report.on_click(_on_run_report)
btn_save_report.on_click(_on_save_report)

display(widgets.VBox([
    box_amp,
    widgets.HBox([ft_flat, ft_grad, ft_mae, ft_r2]),
    widgets.HBox([btn_run_report, prog_report, btn_save_report]),
    out_report,
]))


## Section 3 — Per-epoch navigator

Walk through the rejected epochs one by one. **Montage** (± context) with the flagging cause highlighted
per channel, plus a **detail panel** (PSD + aperiodic fit, band power + 50 Hz ratio, epoch spectrogram,
per-channel metric table). Use the **keep / reject** toggle to override the decision — overrides feed
Section 4.

In [ ]:
dd_filter  = widgets.Dropdown(description='Show:', options=['All rejected'], style={'description_width':'60px'},
                              layout=widgets.Layout(width='260px'))
dd_spectro = widgets.Dropdown(description='Spectro ch:', options=[], style={'description_width':'80px'},
                              layout=widgets.Layout(width='220px'))
sl_ctx     = widgets.IntSlider(description='± context', value=1, min=0, max=3, layout=widgets.Layout(width='240px'))
cb_context = widgets.Checkbox(value=False, description='Show EOG/EMG context', indent=False,
                              layout=widgets.Layout(width='220px'))
sl_epoch   = widgets.IntSlider(description='Index', value=0, min=0, max=0, continuous_update=False,
                               layout=widgets.Layout(width='400px'))
btn_prev   = widgets.Button(description='◀ Prev', layout=widgets.Layout(width='90px'))
btn_next   = widgets.Button(description='Next ▶', layout=widgets.Layout(width='90px'))
tgl_keep   = widgets.ToggleButtons(options=[('Keep','keep'), ('Reject','reject')], value='reject',
                                   style={'button_width':'90px'})
lbl_pos    = widgets.HTML('')
btn_run_nav = widgets.Button(description='Start navigator', button_style='primary', icon='play')

def _nav_list():
    P = S['P']; rej = np.where(P['reject_flag'])[0]
    sel = dd_filter.value
    if sel.startswith('method: '):
        m = sel.split('method: ')[1]
        rej = [ei for ei in rej if bool(P['meta'].loc[ei, 'flag_' + m])] if ('flag_' + m) in P['meta'].columns else []
    elif sel.startswith('stage: '):
        st = sel.split('stage: ')[1]
        rej = [ei for ei in rej if P['stages'][ei] == st]
    return list(rej)

def _refresh_filter_options():
    P = S['P']
    opts = ['All rejected']
    opts += [f'method: {m}' for m in P['methods_present']
             if int(P['meta'].loc[P['reject_flag'], 'flag_' + m].sum()) > 0]
    opts += [f'stage: {st}' for st in L.ordered_present_stages(P['stages'][P['reject_flag']], S['custom_stages'])]
    dd_filter.options = opts
    dd_filter.value = 'All rejected'

def _current_ei():
    nav = S.get('nav_list', [])
    if not nav:
        return None
    i = min(sl_epoch.value, len(nav) - 1)
    return nav[i]

def _render():
    with out_nav:
        clear_output(wait=True)
        try:
            nav = S.get('nav_list', [])
            if not nav:
                print('No rejected epochs for this filter.'); return
            ei = _current_ei()
            thr = _collect_thresholds()
            P = S['P']
            fr = 'reject' if S['final_reject'][ei] else 'keep'
            tgl_keep.unobserve(_on_toggle, 'value'); tgl_keep.value = fr; tgl_keep.observe(_on_toggle, 'value')
            ov = ' (overridden)' if ei in S['overridden'] else ''
            lbl_pos.value = (f'<b>Epoch {ei}</b> — {sl_epoch.value+1}/{len(nav)} — stage {P["stages"][ei]} '
                             f'— reject_method={P["meta"].loc[ei,"reject_method"]} — decision: <b>{fr}</b>{ov}')
            spi = P['ch_names'].index(dd_spectro.value) if dd_spectro.value in P['ch_names'] else 0
            ctx = S.get('ctx') if cb_context.value else None
            f1 = L.plot_epoch_montage(P, ei, thr, context=int(sl_ctx.value), ctx=ctx); show_fig(f1)
            f2 = L.plot_epoch_detail(P, ei, S['freqs'], S['psds'], thr, spectro_ch_idx=spi, fmin=S['pinfo']['fit_range'][0]); show_fig(f2)
        except Exception as e:
            print(f'Render error: {e}')

def _on_run_nav(b):
    try:
        if 'P' not in S:
            with out_nav: clear_output(); print('Load a participant first (Section 1).'); return
        P = S['P']
        if S['freqs'] is None:
            S['freqs'], S['psds'] = L.compute_psds(P['epochs'], fmax=S['pinfo']['fit_range'][1])
        dd_spectro.options = P['ch_names']; dd_spectro.value = P['ch_names'][0]
        _refresh_filter_options()
        S['nav_list'] = _nav_list()
        sl_epoch.max = max(0, len(S['nav_list']) - 1); sl_epoch.value = 0
        _render()
    except Exception as e:
        with out_nav: clear_output(); print(f'Navigator error: {e}')

def _on_filter(chg):
    try:
        S['nav_list'] = _nav_list()
        sl_epoch.max = max(0, len(S['nav_list']) - 1); sl_epoch.value = 0
        _render()
    except Exception as e:
        with out_nav: print(f'Filter error: {e}')

def _step(delta):
    def _f(b):
        sl_epoch.value = int(np.clip(sl_epoch.value + delta, 0, sl_epoch.max))
    return _f

def _on_toggle(chg):
    try:
        ei = _current_ei()
        if ei is None: return
        S['final_reject'][ei] = (chg['new'] == 'reject')
        if S['final_reject'][ei] != bool(S['P']['reject_flag'][ei]):
            S['overridden'].add(ei)
        else:
            S['overridden'].discard(ei)
        _render()
    except Exception as e:
        with out_nav: print(f'Toggle error: {e}')

btn_run_nav.on_click(_on_run_nav)
dd_filter.observe(_on_filter, 'value')
sl_epoch.observe(lambda chg: _render(), 'value')
sl_ctx.observe(lambda chg: _render(), 'value')
cb_context.observe(lambda chg: _render(), 'value')
dd_spectro.observe(lambda chg: _render(), 'value')
btn_prev.on_click(_step(-1)); btn_next.on_click(_step(+1))
tgl_keep.observe(_on_toggle, 'value')

display(widgets.VBox([
    widgets.HBox([btn_run_nav, dd_filter, dd_spectro, sl_ctx, cb_context]),
    widgets.HBox([btn_prev, sl_epoch, btn_next]),
    widgets.HBox([widgets.HTML('<b>Decision:</b>'), tgl_keep]),
    lbl_pos, out_nav,
]))


## Section 4 — Manual override & save

Review the final keep/reject decision (with your overrides) and export the validated epochs. Saves
`*_clean-epo.fif` (kept epochs only), a per-epoch `*_epoch_rejection_reviewed.tsv`, and a
`*_qc2b_review_log.tsv` (one row per overridden epoch), all under
`derivatives/clean_epo_manual/`. **Tool-6 outputs are never modified.**

In [ ]:
btn_review = widgets.Button(description='Refresh review', icon='eye')
btn_save   = widgets.Button(description='Save clean-epo + logs', button_style='success', icon='save')

def _on_review(b):
    with out_review:
        clear_output(wait=True)
        try:
            if 'P' not in S:
                print('Load a participant first (Section 1).'); return
            P = S['P']; fr = S['final_reject']
            n = len(fr); orig = P['reject_flag']
            rescued = int(np.sum(orig & ~fr)); added = int(np.sum(~orig & fr))
            print(f'Epochs: {n} | final keep: {int((~fr).sum())} | final reject: {int(fr.sum())} '
                  f'| rescued: {rescued} | newly rejected: {added} | overrides: {len(S["overridden"])}')
            fig = L.plot_review_strip(P, fr, S['overridden'], custom_stages=S['custom_stages'])
            show_fig(fig)
        except Exception as e:
            print(f'Review error: {e}')

def _on_save(b):
    with out_review:
        try:
            P = S['P']; fr = np.asarray(S['final_reject'], dtype=bool)
            fid = S['fid']
            # Write manual outputs into derivatives/clean_epo_manual/<edf_subtree>/ (kept separate from
            # 7bis's clean_epo_auto/; mirrors the tool-6 subtree). Fall back to the source folder if the
            # derivatives root is unknown.
            src_folder = Path(S['folder'])
            deriv_root = Path(S.get('deriv_root') or src_folder)
            try:
                subtree = src_folder.relative_to(deriv_root)
            except Exception:
                subtree = Path('.')
            # Tool 6 writes epochs under derivatives/raw_epo/<subtree>/; strip the leading 'raw_epo'
            # so clean_epo_manual/ mirrors the EDF <subtree> directly (pre-raw_epo layouts pass through).
            if subtree.parts and subtree.parts[0] == 'raw_epo':
                subtree = Path(*subtree.parts[1:]) if len(subtree.parts) > 1 else Path('.')
            folder = deriv_root / 'clean_epo_manual' / subtree
            folder.mkdir(parents=True, exist_ok=True)
            meta = P['meta'].copy()
            meta['manual_override'] = [i in S['overridden'] for i in range(len(fr))]
            meta['final_reject'] = fr
            epochs = P['epochs'].copy(); epochs.metadata = meta
            keep_idx = np.where(~fr)[0]
            clean = epochs[keep_idx]
            clean.save(str(folder / f'{fid}_clean-epo.fif'), overwrite=True, verbose=False)
            meta.to_csv(folder / f'{fid}_epoch_rejection_reviewed.tsv', sep='\t', index=False)
            rows = []
            for i in sorted(S['overridden']):
                action = 'rescued' if (P['reject_flag'][i] and not fr[i]) else \
                         ('added' if (not P['reject_flag'][i] and fr[i]) else 'unchanged')
                rows.append({'epoch_idx': int(i), 'stage': P['stages'][i],
                             'orig_reject': bool(P['reject_flag'][i]), 'final_reject': bool(fr[i]),
                             'action': action})
            pd.DataFrame(rows, columns=['epoch_idx','stage','orig_reject','final_reject','action']) \
                .to_csv(folder / f'{fid}_qc2b_review_log.tsv', sep='\t', index=False)
            print(f'Saved {len(keep_idx)} clean epochs → {folder / (fid + "_clean-epo.fif")}')
            print(f'  + {fid}_epoch_rejection_reviewed.tsv  +  {fid}_qc2b_review_log.tsv')
        except Exception as e:
            print(f'Save error: {e}')

btn_review.on_click(_on_review)
btn_save.on_click(_on_save)
display(widgets.VBox([widgets.HBox([btn_review, btn_save]), out_review]))
